# Solutions — Advanced String Functions (Medium 11)

**Datasets:** `sales_customers`, `sales_transactions`

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

customers    = spark.table("samples.bakehouse.sales_customers")
transactions = spark.table("samples.bakehouse.sales_transactions")

## Solution 1 — Normalize Phone Numbers

In [ ]:
# Why: regexp_replace strips non-digits; lpad pads to 10; length gives digit count
result_1 = (
    customers
    .withColumn("normalised_phone",
        F.lpad(F.regexp_replace("phone_number", r"[^0-9]", ""), 10, "0"))
    .withColumn("digit_count",
        F.length(F.regexp_replace("phone_number", r"[^0-9]", "")))
    .select("customerID","phone_number","normalised_phone","digit_count")
)

## Solution 2 — Email Validation (Count Valid vs Invalid)

In [ ]:
# Why: .rlike checks the pattern; aggregate to 2-row summary
result_2 = (
    customers
    .withColumn("is_valid_email",
        F.col("email_address").rlike(r"^[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}$"))
    .groupBy("is_valid_email")
    .agg(F.count("*").alias("customer_count"))
)

## Solution 3 — Top 10 Area Codes by Customer Count

In [ ]:
result_3 = (
    customers
    .withColumn("digits_only", F.regexp_replace("phone_number", r"[^0-9]", ""))
    .withColumn("area_code", F.substring("digits_only", 1, 3))
    .filter(F.length("area_code") == 3)
    .groupBy("area_code")
    .agg(F.count("*").alias("customer_count"))
    .orderBy(F.col("customer_count").desc())
    .limit(10)
)

## Solution 4 — Standardize Product Names (initcap + trim)

In [ ]:
result_4 = (
    transactions
    .select(F.col("product").alias("original_product"))
    .distinct()
    .withColumn("clean_product", F.initcap(F.trim("original_product")))
    .orderBy("original_product")
)

## Solution 5 — Format Customer Display Label

In [ ]:
result_5 = (
    customers
    .withColumn(
        "display_label",
        F.concat_ws(" ",
            F.initcap("first_name"), F.initcap("last_name"),
            F.concat(F.lit("("), F.col("country"), F.lit(")"))
        )
    )
    .select("customerID","display_label","email_address")
)

## Solution 6 — Products Containing Digits

In [ ]:
result_6 = (
    transactions
    .withColumn("contains_digit", F.col("product").rlike(r"[0-9]"))
    .groupBy("product","contains_digit")
    .agg(F.count("*").alias("transaction_count"))
    .orderBy("contains_digit", F.col("transaction_count").desc())
)

## Solution 7 — Mask Email Address

In [ ]:
# Why: keep first char + mask username + keep full domain
result_7 = (
    customers
    .withColumn("username", F.split("email_address", "@")[0])
    .withColumn("domain",   F.split("email_address", "@")[1])
    .withColumn("masked_username",
        F.concat(
            F.substring("username", 1, 1),
            F.regexp_replace(F.substring("username", 2, 100), ".", "*")
        )
    )
    .withColumn("masked_email",
        F.concat(F.col("masked_username"), F.lit("@"), F.col("domain")))
    .select("customerID","email_address","masked_email")
)